# AdaBoost — Adaptive Boosting with Decision Stumps

## 1. Definition and intuition
AdaBoost builds a sequence of weak learners. After each learner, it increases the weights of examples that were classified incorrectly, so the next learner focuses more on hard cases. Predictions are combined with weights based on each learner's error.

## 2. Binary AdaBoost mathematics
Encode labels as $y_i\in\{-1,+1\}$. At boosting round t, the weighted error is $\epsilon_t=\sum_i D_t(i)\mathbf{1}[y_i\ne h_t(x_i)]$. For a binary classifier, the learner weight is

$$
\alpha_t=\frac12\log\frac{1-\epsilon_t}{\epsilon_t}.
$$

Update observation weights by $D_{t+1}(i)\propto D_t(i)\exp[-\alpha_t y_i h_t(x_i)]$, then normalize to sum to one. Correctly classified observations lose relative weight; mistakes gain relative weight.

## 3. Why it can work and limitations
Under the classic binary setup with weak learners better than chance, the exponential-loss interpretation explains why weighted errors are emphasized. Training error has a bound related to the product of the weak learners' normalization factors. This is not a guarantee of good test performance. Noise and outliers can receive very large weights, and later learners may over-focus on irreducibly difficult points.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, log_loss

SEED = 42
X, y01 = make_classification(n_samples=900, n_features=10, n_informative=5, n_redundant=2,
                             class_sep=1.0, flip_y=0.04, random_state=SEED)
y = np.where(y01 == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)


## 4. A weighted decision stump from scratch

A decision stump is a one-split tree. For each feature and candidate threshold, we consider both polarities and find the prediction with the smallest weighted error. The simple exhaustive search is for learning, not large-scale production use.

In [2]:
class WeightedDecisionStump:
    def fit(self, X, y, sample_weight):
        X = np.asarray(X, dtype=float); y = np.asarray(y)
        w = np.asarray(sample_weight, dtype=float)
        n, d = X.shape
        best = (np.inf, None, None, None)
        for j in range(d):
            vals = np.unique(X[:, j])
            if len(vals) > 1:
                thresholds = (vals[:-1] + vals[1:]) / 2
            else:
                thresholds = vals
            for threshold in thresholds:
                base = np.where(X[:, j] <= threshold, -1, 1)
                for polarity in (1, -1):
                    pred = base * polarity
                    err = float(np.sum(w[pred != y]))
                    if err < best[0]:
                        best = (err, j, float(threshold), polarity)
        self.weighted_error_, self.feature_, self.threshold_, self.polarity_ = best
        return self

    def predict(self, X):
        base = np.where(np.asarray(X)[:, self.feature_] <= self.threshold_, -1, 1)
        return base * self.polarity_

def fit_adaboost_stumps(X, y, n_estimators=30):
    X = np.asarray(X, dtype=float); y = np.asarray(y)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
        raise ValueError("X must be non-empty 2D and y matching 1D")
    if set(np.unique(y)) != {-1, 1}:
        raise ValueError("Labels must be -1 and +1")
    D = np.full(len(y), 1 / len(y))
    models, alphas, errors = [], [], []
    for _ in range(n_estimators):
        stump = WeightedDecisionStump().fit(X, y, D)
        err = np.clip(stump.weighted_error_, 1e-12, 1 - 1e-12)
        if err >= 0.5:
            break
        alpha = 0.5 * np.log((1 - err) / err)
        pred = stump.predict(X)
        D *= np.exp(-alpha * y * pred)
        D /= D.sum()
        models.append(stump); alphas.append(alpha)
        errors.append(float(np.sum(D[pred != y])))
        if err <= 1e-12:
            break
    return models, np.asarray(alphas), errors

def predict_adaboost(models, alphas, X):
    if len(models) == 0:
        return np.ones(len(X), dtype=int)
    score = sum(a*m.predict(X) for m, a in zip(models, alphas))
    return np.where(score >= 0, 1, -1)

models, alphas, train_weighted_errors = fit_adaboost_stumps(X_train, y_train, n_estimators=25)
pred_scratch = predict_adaboost(models, alphas, X_test)
print("Stumps trained:", len(models))
print("Scratch AdaBoost accuracy:", accuracy_score(y_test, pred_scratch))


Stumps trained: 25
Scratch AdaBoost accuracy: 0.7155555555555555


## 5. Compare with scikit-learn

The precise stumps and tie-breaking can differ. We compare held-out accuracy. `AdaBoostClassifier` parameter names have changed across scikit-learn versions, so this cell uses the current `estimator=` API.

In [3]:
sk = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED),
    n_estimators=60, learning_rate=0.5, random_state=SEED
).fit(X_train, y_train)
pred_sk = sk.predict(X_test)
print("scikit-learn AdaBoost accuracy:", accuracy_score(y_test, pred_sk))
assert pred_scratch.shape == y_test.shape and pred_sk.shape == y_test.shape


scikit-learn AdaBoost accuracy: 0.7155555555555555


## 6. Hyperparameters and best practices

- `n_estimators`: number of boosting rounds; more rounds can help until overfitting or noisy examples dominate.
- `learning_rate`: scales each learner's contribution; smaller values often require more rounds.
- `estimator`: shallow learners are typical weak learners.
- Use validation to tune rounds and learning rate.
- Monitor mislabeled examples, outliers and sample weights.
- For multiclass tasks, AdaBoost uses variants of the binary formulation; do not directly reuse this scratch binary implementation.

## 7. Complexity and scaling

The exhaustive stump fitter checks thresholds for every feature at every round and is intentionally slow; its worst-case cost can be much higher than a sorted, optimized stump search. If there are T rounds, each base-learner fit and prediction cost multiplies across rounds. AdaBoost is sequential: round t depends on weights from round t−1, so rounds are not freely parallelizable.

For large datasets, use optimized library implementations, reduce candidate thresholds when appropriate, and profile the number of rounds. GPU acceleration is not inherent; the sequential tree structure often limits gains.

In [4]:
assert np.isfinite(pred_scratch).all()
assert set(np.unique(pred_scratch)).issubset({-1, 1})
assert len(models) == len(alphas)
try:
    fit_adaboost_stumps(np.array([1,2,3]), np.array([-1,1,-1]))
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid X shape to raise")
print("All AdaBoost checks passed.")


All AdaBoost checks passed.
